# DataRobot + Neo4j Agent — End-to-End Notebook

This notebook covers the full lifecycle of the **Neo4j Research Agent** on DataRobot:

| Section | What it does |
|---|---|
| **1. Setup** | Install deps, load credentials |
| **2. Local smoke test** | Call `chat()` directly — no DataRobot required |
| **3. Deploy** | Run `infra/agent.py deploy` to push to DataRobot automatically |
| **4. Live endpoint test** | Call the deployed Chat completion API |
| **5. Multi-turn memory** | Show conversation continuity via NAMS memory |
| **6. MCP tools** | Verify MCP dynamic tools are being used |
| **7. Batch queries** | Run several research questions end-to-end |

> **Run sections 1–2 to test locally. Run 3–7 once deployed to DataRobot.**


---
## 1. Setup

Install requirements and load credentials from `.env`.  
Copy `.env.example` → `.env` and fill in your keys before running.


In [ ]:
%pip install -q -r requirements.txt


In [ ]:
from __future__ import annotations
import os, json, urllib.request, urllib.error
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv(Path(".env"))

# ── Credentials check ──────────────────────────────────────────────────────
required = ["OPENAI_API_KEY", "NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD"]
missing  = [k for k in required if not os.environ.get(k)]
if missing:
    raise EnvironmentError(f"Missing required env vars: {missing}\nCopy .env.example → .env and fill them in.")

dr_endpoint = os.environ.get("DATAROBOT_ENDPOINT", "").rstrip("/")
dr_token    = os.environ.get("DATAROBOT_API_TOKEN", "")
dr_ready    = bool(dr_endpoint and dr_token)

print("Local credentials:  OK")
print(f"DataRobot access:   {'OK  → ' + dr_endpoint if dr_ready else 'NOT configured (sections 3–7 will be skipped)'}")


---
## 2. Local smoke test

Calls `chat()` directly in-process — no DataRobot deployment needed.
Good for verifying agent logic and tool calls before deploying.


In [ ]:
from agent.custom import chat

def ask_local(question: str, history: list | None = None) -> str:
    """Call the local chat() entrypoint and return the assistant reply."""
    messages = (history or []) + [{"role": "user", "content": question}]
    request  = {"model": os.environ.get("OPENAI_MODEL", "gpt-4o-mini"), "messages": messages}
    response = chat(request, model=request["model"])
    return response["choices"][0]["message"]["content"]

reply = ask_local("Give me a brief profile of Google: key people, industries, and one recent development.")
display(Markdown(reply))


In [ ]:
# Verify tools are firing — look for tool usage in the response
reply = ask_local("Which companies in the Neo4j database operate in the cloud computing industry?")
display(Markdown(reply))


---
## 3. Deploy to DataRobot

Runs the automated deployment script. Requires `DATAROBOT_ENDPOINT` and `DATAROBOT_API_TOKEN` in `.env`.

The script will:
1. Package agent files → `dist/neo4j_datarobot_agent.zip`
2. Create a custom model on DataRobot (`targetType: agenticWorkflow`)
3. Upload all files and wait for the container build
4. Register in the Model Registry
5. Create a live deployment and print the Chat endpoint URL

> **Skip this cell** if you have already deployed — just fill in `DEPLOYMENT_ID` in section 4.


In [ ]:
if not dr_ready:
    print("Skipping — DATAROBOT_ENDPOINT / DATAROBOT_API_TOKEN not set.")
else:
    import subprocess, sys
    result = subprocess.run([sys.executable, 'infra/agent.py', 'deploy'])
    if result.returncode != 0:
        print("\n⚠️  Deploy exited with errors — check DATAROBOT_ENDPOINT/DATAROBOT_API_TOKEN.")
    else:
        print("\n✅ Deployment succeeded.")
    print("\n✅ Copy the Deployment ID printed above into the cell below.")


---
## 4. Live endpoint test

Set `DEPLOYMENT_ID` to the ID shown after deployment (or find it in the DataRobot UI under **Deployments**).

The agent exposes an **OpenAI-compatible Chat Completions API** at:
```
{DATAROBOT_ENDPOINT}/api/v2/deployments/{DEPLOYMENT_ID}/chatCompletions/
```


In [ ]:
# ── Fill in your Deployment ID here ──────────────────────────────────────
DEPLOYMENT_ID = os.environ.get("DR_DEPLOYMENT_ID", "")  # set in .env or override here
# DEPLOYMENT_ID = "your-deployment-id-here"

if not dr_ready or not DEPLOYMENT_ID:
    print("Skipping — set DATAROBOT_ENDPOINT, DATAROBOT_API_TOKEN and DR_DEPLOYMENT_ID.")
else:
    CHAT_URL = f"{dr_endpoint}/api/v2/deployments/{DEPLOYMENT_ID}/chatCompletions/"
    print(f"Chat endpoint: {CHAT_URL}")


In [ ]:
def ask_deployed(question: str, history: list | None = None, deployment_id: str | None = None) -> str:
    """Call the live DataRobot deployment Chat endpoint."""
    deployment_id = deployment_id or globals().get("DEPLOYMENT_ID", "")
    if not dr_ready or not deployment_id:
        return "[Skipped — DataRobot not configured]"

    url      = f"{dr_endpoint}/api/v2/deployments/{deployment_id}/chatCompletions/"
    messages = (history or []) + [{"role": "user", "content": question}]
    payload  = json.dumps({"model": "gpt-4o-mini", "messages": messages}).encode()

    req = urllib.request.Request(
        url, data=payload,
        headers={
            "Authorization": f"Token {dr_token}",
            "Content-Type":  "application/json",
            "Accept":         "application/json",
        },
        method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            return json.loads(resp.read())["choices"][0]["message"]["content"]
    except urllib.error.HTTPError as exc:
        return f"HTTP {exc.code}: {exc.read().decode(errors='replace')}"


reply = ask_deployed("Give me a competitive snapshot of Microsoft: profile, key relationships, and recent news.")
display(Markdown(reply))


---
## 5. Multi-turn memory (NAMS)

Shows conversation continuity. If `MEMORY_API_KEY` and `MEMORY_WORKSPACE_ID` are configured,
the agent stores context in **Neo4j Agent Memory Service (NAMS)** and recalls it across turns.

The `user` field in the request is used as the session / conversation ID.


In [ ]:
SESSION_ID = "notebook-demo-001"   # change to test different sessions

turn1 = ask_deployed(
    "Tell me about Apple Inc — focus on their key executives and recent acquisitions.",
)
display(Markdown("**Turn 1 — Research Apple:**"))
display(Markdown(turn1))


In [ ]:
turn2 = ask_deployed(
    "What industries do they compete in, and who are their closest rivals in the graph?",
    history=[
        {"role": "user",      "content": "Tell me about Apple Inc — focus on their key executives and recent acquisitions."},
        {"role": "assistant", "content": turn1},
    ],
)
display(Markdown("**Turn 2 — Follow-up (tests conversation history):**"))
display(Markdown(turn2))


---
## 6. MCP dynamic tools

If `MCP_SERVER_URL` is configured as a Runtime Parameter on the deployment,
the agent discovers and calls tools from the MCP server alongside the 10 built-in Neo4j tools.

The cell below asks a question that would naturally trigger MCP tools (if available).


In [ ]:
mcp_reply = ask_deployed(
    "Using any available tools, what can you tell me about the relationship between "
    "OpenAI and Microsoft in the knowledge graph?",
)
display(Markdown("**MCP + Neo4j tools combined:**"))
display(Markdown(mcp_reply))


---
## 7. Batch research queries

Run several industry research questions to validate the deployed agent end-to-end.


In [ ]:
DEMO_QUESTIONS = [
    "List 5 companies in the AI industry from the graph with a one-sentence description each.",
    "Who are the top 3 most connected people in the knowledge graph and what companies are they linked to?",
    "Summarise the latest news articles about Amazon from the graph.",
    "What industries have the most companies in the Neo4j database?",
]

for i, q in enumerate(DEMO_QUESTIONS, 1):
    print(f"\n{'='*60}")
    print(f"Q{i}: {q}")
    print("="*60)
    reply = ask_deployed(q)
    display(Markdown(reply))


---
## Useful links

| Resource | URL |
|---|---|
| DataRobot Deployments | `{DATAROBOT_ENDPOINT}/deployments/` |
| DataRobot Workshop (model files) | Registry → Workshop (left sidebar) |
| NAMS memory dashboard | https://memory.neo4jlabs.com |
| neo4j-mcp-official | https://neo4j-mcp-official-1008050579172.us-central1.run.app/mcp |
| Agent source code | `datarobot/agent/` |
| Deploy script | `python infra/agent.py deploy` |
